In [ ]:
# 1. تثبيت المكتبة (لو مش متثبتة)
!pip install ultralytics 

In [ ]:
# 2. استدعاء المكتبة
from ultralytics import YOLO

# 3. تحميل موديل مبدئي (Pre-trained) عشان يسرع التدريب ويحسن الدقة
# بنستخدم نسخة الـ nano (yolov8n.pt) لأنها سريعة جداً ومناسبة للوحات، ممكن تستخدم yolov8s.pt لو عايز دقة أعلى شوية
model = YOLO('yolov8s.pt') 

# 4. بدء التدريب
# ⚠️ مهم جداً: غير مسار ملف data.yaml بناءً على اسم الداتا اللي رفعتها على كاجل
yaml_path = '/kaggle/input/datasets/saifhany259/ocr-data/OCR Car Plates Recognition.v1/data.yaml' 

results = model.train(
    data=yaml_path,
    epochs=60,        # عدد مرات المرور على الداتا (50 رقم كويس كبداية، ممكن تزوده لـ 100 لو لقيت الدقة لسه بتتحسن)
    imgsz=640,        # حجم الصورة اللي الموديل هيتدرب عليها (الافتراضي 640)
    batch=16,         # عدد الصور اللي بتدخل للميموري في المرة الواحدة
    device=0,         # 0 معناها استخدام الـ GPU
    project='License_Plate_Project', # اسم الفولدر اللي هيتحفظ فيه النتائج
    name='train_1'    # اسم التجربة
)

print("تم التدريب بنجاح!")

In [ ]:
import cv2
import easyocr
from ultralytics import YOLO
import os  

# ==========================================
# 1. الإعدادات والتهيئة
# ==========================================
model_path = '/kaggle/working/runs/detect/License_Plate_Project/train_1/weights/best.pt'
yolo_model = YOLO(model_path)

# تهيئة EasyOCR
reader = easyocr.Reader(['en'], gpu=True)

# مسار الفيديو
video_path = '/kaggle/input/datasets/saifhany259/ocr-video/Test video.MP4' 
cap = cv2.VideoCapture(video_path)

# ملف لحفظ النتائج النصية
output_text_file = open('detected_plates_final.txt', 'w')

# إنشاء مجلد لحفظ صور اللوحات
plates_folder = 'saved_plates'
if not os.path.exists(plates_folder):
    os.makedirs(plates_folder)

# إعدادات الفيديو الناتج
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
out_video = cv2.VideoWriter('output_result.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

frame_number = 0
plate_counter = 1 

print("بدء معالجة الفيديو بنظام الـ Tracking...")

# ==========================================
# 2. حلقة المعالجة الرئيسية (استخدام Track بدلاً من Predict)
# ==========================================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_number += 1
    
    # استخدام .track بدلاً من .predict لتقليل الرعشة (persist=True للمحافظة على الـ ID عبر الفريمات)
    results = yolo_model.track(frame, conf=0.5, persist=True, tracker="bytetrack.yaml", verbose=False)
    
    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            
            # قص اللوحة (مع هامش بسيط)
            plate_crop = frame[max(0, y1-5) : min(height, y2+5), 
                               max(0, x1-5) : min(width, x2+5)]
            
            if plate_crop.size != 0:
                # تكبير اللوحة لزيادة دقة الـ OCR
                zoomed_plate = cv2.resize(plate_crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
                
                # تحويل المكبرة لرمادي
                gray_plate = cv2.cvtColor(zoomed_plate, cv2.COLOR_BGR2GRAY)
                ocr_results = reader.readtext(gray_plate)
                
                detected_text = ""
                for (bbox, text, prob) in ocr_results:
                    if prob > 0.4: 
                        detected_text += text + " "
                
                clean_text = detected_text.replace(" ", "")
                
                # شرط الفلترة: 6 خانات فأكثر
                if len(clean_text) >= 6:
                    
                    # 1. رسم المربع الأخضر على الفريم الأصلي
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # 2. خاصية الـ PiP (وضع اللوحة المكبرة في زاوية الفيديو)
                    thumbnail = cv2.resize(zoomed_plate, (200, 60))
                    frame[10:70, 10:210] = thumbnail
                    
                    # 3. حفظ صورة اللوحة
                    img_name = f"plate_{plate_counter}_frame_{frame_number}.jpg"
                    img_path = os.path.join(plates_folder, img_name)
                    cv2.imwrite(img_path, zoomed_plate)
                    plate_counter += 1
                    
                    # 4. تسجيل النتيجة
                    final_text = detected_text.strip()
                    print(f"Frame {frame_number}: Plate [{final_text}]")
                    output_text_file.write(f"Frame {frame_number}: {final_text}\n")
                    
                    # 5. كتابة النص على الفيديو
                    cv2.putText(frame, final_text, (x1, y1 - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    out_video.write(frame)

# ==========================================
# 3. إنهاء العمليات
# ==========================================
cap.release()
out_video.release()
output_text_file.close()

print(f"تم الانتهاء بنجاح! تم حفظ النتائج في 'output_result.mp4'")

In [ ]:
import cv2
import easyocr
from ultralytics import YOLO
import os  

# ==========================================
# 1. الإعدادات والتهيئة
# ==========================================
model_path = '/kaggle/working/runs/detect/License_Plate_Project/train_1/weights/best.pt'
yolo_model = YOLO(model_path)
reader = easyocr.Reader(['en'], gpu=True)

video_path = '/kaggle/input/datasets/saifhany259/ocr-video/Test video.MP4' 
cap = cv2.VideoCapture(video_path)

# --- تعريف الأبعاد هنا (مهم جداً قبل استخدام width و height) ---
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))

# --- حل مشكلة الخط: جعله في منتصف الشاشة دائماً ---
line_y = 770 
is_saved = False 

# --- إضافة مجلد حفظ الصور ---
plates_folder = 'saved_plates'
if not os.path.exists(plates_folder):
    os.makedirs(plates_folder)
plate_counter = 1

out_video = cv2.VideoWriter('output_result.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

# ==========================================
# 2. حلقة المعالجة
# ==========================================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    # رسم الخط 
    cv2.line(frame, (0, line_y), (width, line_y), (0, 0, 255), 3)
    
    # تقليل conf لـ 0.4 لضمان التقاط اللوحات
    results = yolo_model.track(frame, conf=0.20, persist=True, tracker="bytetrack.yaml", verbose=False)
    
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            center_y = int((y1 + y2) / 2)
            
            # منطق الـ Tripwire: توسيع النطاق لـ 30 عشان السيارات السريعة
            if abs(center_y - line_y) < 30 and not is_saved:
                plate_crop = frame[y1:y2, x1:x2]
                
                if plate_crop.size != 0:
                    zoomed_plate = cv2.resize(plate_crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
                    
                    # عملية القراءة
                    gray = cv2.cvtColor(zoomed_plate, cv2.COLOR_BGR2GRAY)
                    ocr_results = reader.readtext(gray)
                    
                    # --- إضافة جزء الفلترة والحفظ ---
                    detected_text = ""
                    for (bbox, text, prob) in ocr_results:
                        if prob > 0.4: 
                            detected_text += text + " "
                    
                    clean_text = detected_text.replace(" ", "")
                    
                    if len(clean_text) >= 6:
                        # 1. حفظ صورة اللوحة
                        img_path = os.path.join(plates_folder, f"plate_{plate_counter}.jpg")
                        cv2.imwrite(img_path, zoomed_plate)
                        plate_counter += 1
                        
                        # 2. رسم المربع والنص على الفيديو الناتج لتوثيق العملية
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cv2.putText(frame, detected_text.strip(), (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
                        
                        print(f"تم التقاط وحفظ اللوحة بنجاح: {detected_text.strip()}")

                    is_saved = True 
            
            elif abs(center_y - line_y) > 50:
                is_saved = False 

    out_video.write(frame)

# 3. إنهاء العمليات
cap.release()
out_video.release()

# هذه الجملة ستظهر في الكونسول لتؤكد لك نجاح العملية
print("تمت المعالجة بنجاح! ملف الفيديو الناتج 'output_result.mp4' جاهز الآن للمشاهدة.")